# **Pairwise Sequence Alignment**

**Guided teaching edition — original lesson retained**

**Learning route:** global alignment → local alignment → simulated sequencing reads → read mapping → majority-vote SNP calling.

In each exercise, read the **Task**, **Hints**, and **Expected outcome** before editing the `# TODO` lines. Suggested solutions are provided as **commented code** to reveal during teaching. The original explanations, formula, examples, and core exercise structure are preserved.

> **Course dependencies:** The original lesson imports a local `alignment` module and refers to `images/nw_matrix.png`. Keep the course's `alignment.py` and `images/` folder alongside this notebook when available. The mathematical worked example below does not need these files.

# The Needleman-Wunsch Algorithm

### Before the formula: what are we computing?

Suppose `x = "AG"` and `y = "A"`. A possible end-to-end alignment is:

```text
X: A G
Y: A -
   |  
```

With **match = +1**, **mismatch = −1**, **gap = −1**, its score is `+1 − 1 = 0`.

A **dynamic programming (DP) matrix** finds the highest-scoring alignment without enumerating every complete alignment. A cell `F(i,j)` is the **best score for aligning the first `i` characters of X with the first `j` characters of Y**. In particular, `(0,0)` means two empty prefixes.

| Symbol | Meaning | Example |
|---|---|---|
| `i` | Number of X characters already considered (matrix row) | `i=2` means `AG` |
| `j` | Number of Y characters already considered (matrix column) | `j=1` means `A` |
| `xᵢ`, `yⱼ` | Current characters at those prefix ends | `x₂=G`, `y₁=A` (mathematical 1-based subscripts) |
| `s(xᵢ,yⱼ)` | Substitution score: +1 if equal, −1 otherwise | `s(A,A)=+1` |
| `d` | Penalty for introducing one gap | `d=−1` |
| `F(i,j)` | Best score for these two prefixes | `F(1,1)=1` |

**Read the three terms as three choices for the *last alignment column*:**

| Candidate | Where the score comes from | Alignment column added |
|---|---|---|
| `F(i−1,j−1) + s(xᵢ,yⱼ)` | diagonal ↖ | `xᵢ` aligned with `yⱼ` (match/mismatch) |
| `F(i−1,j) + d` | above ↑ | `xᵢ` aligned with `-` (gap in Y) |
| `F(i,j−1) + d` | left ← | `-` aligned with `yⱼ` (gap in X) |

`max` simply chooses whichever of these three possibilities produces the greatest score. **The original formula is shown next, unchanged.**

The Needleman-Wunsch algorithm finds the optimal *global* alignment (end-to-end) between two sequences.
It consists of the following steps:

1. Construct the grid
2. Choose a scoring system (here we use the original scoring system assigning a score $s$ of 1 to a match, a score $s$ of -1 to a mismatch, and a score $d$ of -1 to a gap/indel)
3. Fill in the scoring matrix $F$ (and construct the pointer matrix on the way)
4. Use the pointer matrix to perform a traceback to recover the optimal warping path

The scores in the scoring matrix $F$ are computed using the already existing scores in the left, top-left, and top cells:

$$
F(i,j)=\max
\begin{cases}
    F(i-1,j-1)+s(x_i,y_i),\\
    F(i-1,j)+d,\\
    F(i,j-1)+d.
\end{cases}
$$

The following image illustrates the scoring and pointer matrices.

<img src="images/nw_matrix.png" width="500" />

### Worked example: fill the entire matrix by hand

For `X = AG`, `Y = A`, the matrix has **3 rows** (`empty`, `A`, `AG`) and **2 columns** (`empty`, `A`).

**Step 1 — initialize empty-prefix alignments.** Each leading gap costs −1:

| X prefix \ Y prefix | empty (j=0) | A (j=1) |
|---|---:|---:|
| empty (i=0) | 0 | −1 |
| A (i=1) | −1 | ? |
| AG (i=2) | −2 | ? |

**Step 2 — compute `F(1,1)` (align `A` with `A`).**

- Diagonal: `F(0,0) + s(A,A) = 0 + 1 = 1` **← largest**
- Up: `F(0,1) + d = −1 − 1 = −2`
- Left: `F(1,0) + d = −1 − 1 = −2`

So **`F(1,1) = 1`**, and we record a **↖ pointer**.

**Step 3 — compute `F(2,1)` (align prefixes `AG` and `A`).**

- Diagonal: `F(1,0) + s(G,A) = −1 − 1 = −2`
- Up: `F(1,1) + d = 1 − 1 = 0` **← largest**
- Left: `F(2,0) + d = −2 − 1 = −3`

So **`F(2,1) = 0`**, and we record an **↑ pointer**.

| X prefix \ Y prefix | empty | A |
|---|---:|---:|
| empty | 0 | −1 |
| A | −1 | **1 ↖** |
| AG | −2 | **0 ↑** |

**Final global-alignment score = bottom-right value = `0`.**

### Traceback: turn the scores into actual letters

The scores say *how good* each partial alignment is. The **pointers** say *which previous cell* supplied the winning score. We start at the **bottom-right** and follow pointers **backward**:

1. At `F(2,1)`, pointer **↑** means consume X's `G`, insert a gap in Y: write `G / -`. Move to `F(1,1)`.
2. At `F(1,1)`, pointer **↖** means align X's `A` with Y's `A`: write `A / A`. Move to `F(0,0)`.
3. Stop at `(0,0)`. We reconstructed columns **from last to first**, so reverse their order.

```text
Traceback order:     (G,-) ← (A,A)
Final alignment X:    A G
Final alignment Y:    A -
Total score:          +1 -1 = 0
```

**What if two candidates tie?** Multiple optimal alignments can exist; an implementation uses a tie-breaking rule (or reports multiple paths).

We can use the `make_align()` function from our `alignment` module to generate an `align()` function that performs the Needleman-Wunsch algorithm.

In [ ]:
import alignment as aln

x = "GATTACA"
y = "CATACCCA"

align = aln.make_align(mode="global")
x_aln, y_aln, warp_path_x, warp_path_y, scoring_mat = align(seq_a=x, seq_b=y)

### Reading the five outputs: letters versus positions

```python
x_aln, y_aln, warp_path_x, warp_path_y, scoring_mat = align(seq_a=x, seq_b=y)
```

- **`x_aln` and `y_aln`**: the **aligned characters**, including `-` if the algorithm inserts gaps. These are the sequences we want to display.
- **`warp_path_x` and `warp_path_y`**: numerical coordinates describing **where the selected alignment travels in the algorithm's scoring grid**. They are used for plotting or recovering mapping locations.
- **`scoring_mat`**: the DP scores, not the alignment itself.

For instance, `X = ATG` versus `Y = AG` can align as:

```text
Alignment column:     1 2 3
X (aligned letters):   A T G
Y (aligned letters):   A - G
X base index:          0 1 2
Y base index:          0 - 1
```

A **conceptual full DP-grid path** in `(row, column)` coordinates is `(0,0) → (1,1) → (2,1) → (3,2)`; the second step advances X but not Y, which means **a gap in Y**.

**Important distinction:** this *full grid path* is not necessarily the exact array returned by the course's `warp_path_x` / `warp_path_y`. The original notebook does not include `alignment.py`, so we should **inspect the actual returned arrays** instead of assuming an undocumented indexing and endpoint convention. The next diagnostic cell does exactly that, if the course module is available.

In [ ]:
# Optional diagnostic: verify how your course alignment module represents the path.
print("Aligned X:", x_aln)
print("Aligned Y:", y_aln)
print("Path X:", warp_path_x)
print("Path Y:", warp_path_y)
print("Matrix shape:", scoring_mat.shape)

We can also plot the alignment together with the scoring matrix:

In [ ]:
import matplotlib.pyplot as plt  # type: ignore

plt.imshow(X=scoring_mat)
plt.plot(warp_path_y, warp_path_x, "w")
plt.xticks(ticks=range(len(y)), labels=y)
plt.yticks(ticks=range(len(x)), labels=x)
plt.show()

# The Smith-Waterman Algorithm

The Smith-Waterman algorithm finds the optimal *local* alignment between two sequences.
It is a variation of the Needleman-Wunsch algorithm.

The three main differences are:

|                | Smith-Waterman algorithm                                | Needleman-Wunsch algorithm                                                 |
|----------------|---------------------------------------------------------|----------------------------------------------------------------------------|
| Initialization | First row/column = 0                                    | First row/column subject to gap penalty                                    |
| Scoring        | Negative scores are set to 0                            | Scores can be negative                                                     |
| Traceback      | Begin with the highest score, end when 0 is encountered | Begin with the cell at the lower right of the matrix, end at top left cell |

### Why does local alignment behave differently?

Compare `X = GATTACA` with `Y = NNNNGATTACANNNN`. If we want the **best matching region** rather than forcing the `N` characters to align, we use **Smith–Waterman**.

Smith–Waterman applies the same three candidates but also allows **0**:

$$F(i,j)=\max\big(0,\ F(i-1,j-1)+s(x_i,y_j),\ F(i-1,j)+d,\ F(i,j-1)+d\big).$$

A negative score is replaced by **0** so a poor earlier region does not drag down a promising new local match. Therefore traceback starts at the **highest-scoring cell anywhere** and stops once it reaches **0**. Unlike global alignment, local alignment does **not** have to include both sequences' ends.

**Teaching check:** When mapping a short read to a long reference, do we usually want full-length global alignment between both entire strings? **No** — the read is expected to match just part of the reference.

Again, with our `alignment` module, we can perform the local alignment between the sequences `GATACCA` and `NNNNNNNNNNGATTACANNNNNNNNGATNNNNNNNNN`.

In [ ]:
x = "GATACCA"
y = "NNNNNNNNNNGATTACANNNNNNNNGATNNNNNNNNN"

align = aln.make_align(mode="local")
x_aln, y_aln, warp_path_x, warp_path_y, scoring_mat = align(seq_a=x, seq_b=y)

plt.figure(figsize=(15, 8))
plt.imshow(X=scoring_mat)
plt.xticks(ticks=range(len(y)), labels=y)
plt.yticks(ticks=range(len(x)), labels=x)
plt.plot(warp_path_y, warp_path_x, "w")
plt.show()

# Read Alignment

We can also simulate the alignment of many reads originating from e.g., a bulk DNA sequencing experiment.

### From alignment algorithms to a small sequencing experiment

Imagine we know a DNA **reference sequence** but a sequencing instrument gives us only short fragments, called **reads**. We will simulate those fragments, introduce substitution errors, find where each fragment came from, and then combine evidence across reads to call candidate SNPs.

```text
Reference → sample reads → add noise → local read alignment → pileup → candidate SNPs
```

Each of the next exercises adds one piece of this pipeline.

##### ❓ Sampling reads from a reference sequence

**Task:** Create reads by randomly sampling substrings from `reference_sequence`.

**Expected outcome:** A list of DNA strings, each length 15–25; the current code requests **20 reads** (`n_reads=20`), although the original prose says **5**. You may change `n_reads = 5` to follow that sentence; here we retain the original code value.

**Hints:** `random.randint(a,b)` selects an integer in the inclusive range; `reference_sequence[start:end]` takes a substring with `end` excluded; ensure `end <= len(reference_sequence)`. `reads.append(read)` stores a result; `range_acquired=True` exits the inner loop.

**Think about it:** Why can't a read starting close to the right edge have an arbitrary length?

Simulate the generation of a number of reads, by sampling 5 reads from the reference sequence `GATTTGGGGTTCAAAGCAGTATCGATCAAATAGTAAATCCATTTGTTCAACTCACAGTTTA`.

In [ ]:
import random

reference_sequence = "GATTTGGGGTTCAAAGCAGTATCGATCAAATAGTAAATCCATTTGTTCAACTCACAGTTTA"

n_reads = 20
min_read_len = 15
max_read_len = 25

reads = []

while len(reads) < n_reads:
    range_acquired = False
    while not range_acquired:
        # TODO: choose a valid start and end, slice the reference, and append a read.
        # --- Suggested solution (uncomment to reveal) ---
        # read_len = random.randint(min_read_len, max_read_len)
        # start = random.randint(0, len(reference_sequence) - 1)
        # end = start + read_len
        # if end <= len(reference_sequence):
        #     reads.append(reference_sequence[start:end])
        #     range_acquired = True
        pass

print(reads)

##### ❓ Adding noise to the sampled reads

**Task:** For each base in a read, randomly decide whether to replace it with a different DNA base, and count actual substitutions.

**Expected outcome:** `add_noise()` returns `(noisy_read, n_noisy_bases)`; the two reads have equal length.

**Suggested rule for this teaching version:** `random.randrange(noiselessness) == 0` chooses a base for substitution with probability `1/noiselessness` (so a smaller value introduces *more* noise). This makes the previously unexplained parameter explicit. Always choose a base **different** from the original so the count reflects real changes.

**Hints:** `random.choice(...)` picks an item; `noisy_read += ...` appends a character; `n_noisy_bases += 1` increments a counter. This exercise simulates **substitutions only**, not insertions or deletions.

Add some noise to the sampled reads.

In [ ]:
def add_noise(read: str, noiselessness: int = 4) -> tuple[str, int]:
    """Add noise to a read."""
    choices = ["A", "C", "G", "T"]
    n_noisy_bases = 0
    noisy_read = ""
    for base in read:
        # TODO: sometimes replace 'base' with a different A/C/G/T base.
        # --- Suggested solution (uncomment to reveal) ---
        # if random.randrange(noiselessness) == 0:
        #     alternatives = [candidate for candidate in choices if candidate != base]
        #     noisy_read += random.choice(alternatives)
        #     n_noisy_bases += 1
        # else:
        #     noisy_read += base
        pass
    return noisy_read, n_noisy_bases


noisy_reads = []
for read in reads:
    noisy_read, n_noisy_bases = add_noise(read=read, noiselessness=3)
    print(f"Added {n_noisy_bases:2} noisy base(s): {read:10s} -> {noisy_read:10s}")
    noisy_reads.append(noisy_read)

##### ❓ Aligning reads

**Task:** Create a local aligner and align **each noisy read** against the full `reference_sequence`.

**Expected outcome:** One mapping position and one aligned read per input read, plus a DP matrix visualization.

**Hints:** `aln.make_align(mode="local")` creates the aligner; `align(seq_a=read, seq_b=reference_sequence)` returns **five outputs** in the same order used earlier. Choose variable names `read_aln`, `ref_aln`, `warp_path_read`, `warp_path_ref`, `scoring_mat`.

**Interpretation:** `.join(read_aln)` turns the aligned characters into a string. Clipped bases are the read characters omitted from the selected **local** alignment.

**Implementation note:** The original starter uses `warp_path_ref[0]` as the 0-based mapping start. That is a course-module convention worth confirming against `alignment.py`; it is not guaranteed by every aligner.

Use the `make_align()` function from the `alignment` module to generate an `align()` function to align the noisy reads locally to the reference sequence.

In [ ]:
mapping_positions = []
aligned_reads = []

for read in noisy_reads:
    # TODO: align the current read locally to the reference.
    # --- Suggested solution (uncomment to reveal) ---
    # align = aln.make_align(mode="local")
    # read_aln, ref_aln, warp_path_read, warp_path_ref, scoring_mat = align(
    #     seq_a=read, seq_b=reference_sequence
    # )
    # NOTE: uncomment the solution before running this cell.

    mapping_position = warp_path_ref[0]
    aligned_read = "".join(read_aln)
    mapping_positions.append(mapping_position)
    aligned_reads.append(aligned_read)

    print(f"Read                      : {read}")
    print(f"Mapping position (0-based): {mapping_position}")
    print(f"Aligned read              : {aligned_read}")
    aligned_read_without_gaps = list(filter(lambda elem: elem != "-", read_aln))
    print(f"Aligned read w/o gaps     : {''.join(aligned_read_without_gaps)}")
    print(f"Number of clipped bases   : {len(read) - len(aligned_read_without_gaps)}")
    print(
        f"Reference subsequence     : "
        f"{reference_sequence[warp_path_ref[0] : (warp_path_ref[-1] + 1)]}"
    )

    plt.figure(figsize=(15, 8))
    plt.imshow(X=scoring_mat)
    plt.plot(warp_path_ref, warp_path_read, "w")
    plt.xticks(ticks=range(len(reference_sequence)), labels=reference_sequence)
    plt.yticks(ticks=range(len(read)), labels=read)
    plt.show()

We now have the following set of aligned reads:

In [ ]:
print(f"Reference: {reference_sequence}")
for i, (pos, seq) in enumerate(zip(mapping_positions, aligned_reads)):
    print(f"Read {i:4}: {' ' * pos}{seq}")

# Variant Calling

Variant calling is a crucial step in analyzing DNA sequence data to identify differences or variations in the genetic code of individuals.
This process involves comparing an individual's DNA sequence (in the form of aligned reads) to a reference sequence to pinpoint variations such as single nucleotide polymorphisms (SNPs), insertions, deletions, and structural alterations.

While there exist many sophisticated variant calling algorithms such as [GATK](https://gatk.broadinstitute.org)'s [HaplotypeCaller](https://gatk.broadinstitute.org/hc/en-us/articles/360037225632-HaplotypeCaller), we will here focus on SNPs only and perform a simple majority voting per locus.

### Pileup and majority vote: a tangible example

```text
Reference position:       0 1 2 3 4 5
Reference:                A T G C A T
Read 1:                   A T G T
Read 2:                       G T A T
Read 3:                       G C A T
                                 ^
At position 3, observed bases might be T, T, C.
Majority = T, reference = C → candidate C→T SNP.
```

**A candidate is not proof of a biological variant.** In our toy simulation, substitutions were introduced as sequencing noise, so a majority vote can produce false positives. Real variant callers additionally consider read quality, depth, mapping uncertainty, and genotype models.

##### ❓ SNP calling via majority vote

**Task:** Group all read bases by their reference positions (**pileup**) and then use **majority voting** to find positions where the most common read base differs from the reference.

**Mini example:** `reference = C`, `pileup = TTCT` → majority `T` → candidate `C→T` variant.

**Hints:** `enumerate(seq)` gives `(offset, base)`; `pileups[pos + offset].append(base)` adds a base to one reference position; `Counter(pileup).most_common(1)[0][0]` extracts the most common base.

**Important safeguards:** A locus with **no reads** must not become a SNP. Also, ties and low coverage mean the evidence is weak. This exercise intentionally uses a simple majority vote rather than a full statistical variant caller.

**Simplifying assumption:** The pileup loop below assumes contiguous reference positions per aligned character, suitable for the substitution-only simulation when mapping does not introduce internal gaps. A proper gapped alignment requires tracking reference coordinates, not just using `pos + offset`.

Complete the following code to call SNPs via majority voting.

In [ ]:
from collections import Counter

pileups = [[] for _ in range(len(reference_sequence))]
snps = {}

# Accumulate the pileups.
for pos, seq in zip(mapping_positions, aligned_reads):
    # TODO: add each aligned base to its corresponding pileup.
    # --- Suggested solution (uncomment to reveal) ---
    # for offset, base in enumerate(seq):
    #     if base != "-" and pos + offset < len(reference_sequence):
    #         pileups[pos + offset].append(base)
    pass

# SNP calling via majority voting.
print("POS   REF   MAJ   SNP   PILEUP")
for i, pileup in enumerate(pileups):
    majority_base = ""
    snp = "-"

    # Make a pileup string.
    pileup = "".join(pileup)

    # We can only call variants for non-empty pileups.
    if pileup:
        # TODO: count bases and assign the most frequent one to majority_base.
        # --- Suggested solution (uncomment to reveal) ---
        # majority_base = Counter(pileup).most_common(1)[0][0]
        pass

    # Call it if it's a SNP.
    if pileup and majority_base not in ("", "-") and majority_base != reference_sequence[i]:
        snp = majority_base
        snps[i] = snp

    print(f"{i:3}   {reference_sequence[i]:3}   {majority_base:3}   {snp:3}   {pileup}")

In [ ]:
# List all SNPs.
print("POS   SNP")
for pos, snp in snps.items():
    print(f"{pos:3}   {snp:3}")

## Recap — What should students be able to explain?

| Concept | One-sentence explanation |
|---|---|
| Needleman–Wunsch | Find the best **end-to-end** alignment. |
| `F(i,j)` | Best score for aligning the first `i` and `j` sequence characters. |
| Three recurrence terms | Choose match/mismatch, gap in Y, or gap in X. |
| Traceback | Follow stored best-choice arrows backward to recover aligned letters. |
| Smith–Waterman | Find the best **local** matching region; scores can restart at 0. |
| `x_aln` vs path | Aligned letters versus alignment/grid coordinates. |
| Read mapping | Identify where a short read fits within a reference. |
| Pileup | Collect bases from overlapping reads at each reference position. |
| Majority vote | Report a **candidate** substitution when the most frequent observed base differs from the reference. |

**Instructor discussion prompts:** What happens if two traceback choices tie? What if most reads are wrong at the same site? What if a read maps equally well to two reference locations? What changes if reads contain insertions or deletions?

**Reminder:** The course's external `alignment` module is required for the two algorithm demos and the read-mapping exercise; it is not embedded in this notebook.